<a href="https://colab.research.google.com/github/sudikshyapant/lgmd_for_retinal_fundus/blob/main/lgmd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LGMD — Concept Discovery on Retinal Fundus Images

Post-hoc concept discovery for a **5-class retinal-fundus disease classifier**, via
*Language-Guided Matrix Decomposition* (LGMD, ECCV 2026 #11398).

Scope: one fundus class at a time (the active class, default **cataract**). The backbone is
a **DenseNet-121** fine-tuned to a 5-way head on **ODIR-5K** (classes Normal, Diabetes,
Glaucoma, Cataract, AMD; ~0.755 balanced accuracy). Concept maps come from **BiomedCLIP**.
Baselines (**ICE, CRAFT, FACE**) and the metrics (**Acc, C-Ins, MSE**) follow the paper.
Concepts are read from `concept_vocab.json` (keyed by class) and filtered. Heavy artifacts
are cached.

### Scope

The cells below walk through the pipeline on a **single class** (the active class, default
`cataract`). To run **all 7 classes**, jump to the last section — it loads the backbone +
BiomedCLIP once and loops `config.select_class` over every class, caching per class. A class
that errors is printed and skipped. Qualitative overlays are saved only for
`config.FIGURE_CLASSES` (default: cataract).

In [ ]:
# === Colab setup (skip when running locally) ===
# 1) Clone the repo from GitHub and install dependencies:
# !git clone https://github.com/sudikshyapant/lgmd_for_retinal_fundus
# %cd lgmd_for_retinal_fundus
# !pip -q install -r requirements.txt
#
# 2) No token needed: BiomedCLIP and the DenseNet-121 ImageNet init are public.
#    (config.get_secret still supports HF_TOKEN via Colab Secrets if you later
#     swap in a gated model.)
# 3) Importing config below mounts Google Drive automatically when on Colab.

In [1]:
# !git clone https://github.com/sudikshyapant/lgmd_for_retinal_fundus

Cloning into 'lgmd_for_retinal_fundus'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 90 (delta 38), reused 73 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 8.71 MiB | 34.43 MiB/s, done.
Resolving deltas: 100% (38/38), done.


In [2]:
# %cd lgmd_for_retinal_fundus

/content/lgmd_for_retinal_fundus


In [3]:
!pip -q install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.8 MB/s eta 0:00:00


In [5]:
import os, json, sys

# This notebook belongs to the lgmd_for_retinal_fundus repo. A sibling repo may be cloned
# alongside it and ALSO have a src/config.py, so we anchor on a marker unique to this repo
# (src/lgmd.py) to avoid importing the wrong config. We locate the repo root regardless of
# where the kernel started, cd into it (so config's REPO_ROOT-relative paths resolve), and
# put its src/ first on sys.path.
_MARKERS = (os.path.join("src", "config.py"), os.path.join("src", "lgmd.py"))
_here = os.getcwd()
_candidates = [
    _here,
    os.path.join(_here, "lgmd_for_retinal_fundus"),
    "/content/lgmd_for_retinal_fundus",
    os.path.dirname(_here),
]
for _root in _candidates:
    if all(os.path.isfile(os.path.join(_root, m)) for m in _MARKERS):
        os.chdir(_root)
        sys.path.insert(0, os.path.join(_root, "src"))
        break
else:
    raise RuntimeError(
        "Could not locate the repo root (needs src/config.py and src/lgmd.py). "
        "On Colab, %cd /content/lgmd_for_retinal_fundus first."
    )

import torch
import config
from config import CONFIG, cache_name
import utils, data_utils, model_utils, concepts as concept_mod, clip_maps, lgmd, baselines, metrics, viz

utils.set_seed(CONFIG["seed"])
DEVICE = model_utils.DEVICE
CDIR = CONFIG["cache_dir"]       # heavy/constant artifacts (activations, S, W)
RDIR = CONFIG["results_dir"]     # cached metric tables
bb = CONFIG["backbone"]          # used in figure filenames
print("device:", DEVICE)

Mounted at /content/drive
device: cuda


### A note on hyperparameters

A few values are best-guess estimates, not paper-specified: red-circle radius and stroke
width, PGD iteration counts, CRAFT depth, FACE settings, and the choice of C-Ins metric.
They are marked `[suppl]` in `config.py` because the supplementary material describes them
only qualitatively. The concept-filtering rules (A1.3/A1.4) and the CLIP red-circle
prompting setup (A1.6) are followed exactly; only the items listed above are estimates.

## 0. Dataset — ODIR-5K, 5 classes (download + prepare)

The classifier is trained on **ODIR-5K** (Ocular Disease Intelligent Recognition), restricted
to five single-label classes: **Normal, Diabetes (DR), Glaucoma, Cataract, AMD** (codes
N / D / G / C / A). ODIR-5K ships as a flat image folder plus a per-eye label table, **not**
an ImageFolder tree, so `odir_prep.prepare` converts it once: it reads each eye's diagnostic
keywords, keeps only images that are unambiguously one of the five classes (mixed / other
diagnoses are dropped), and writes a stratified **70/15/15** `train/ val/ test` split under
`CONFIG["data_root"]`.

The cell below downloads ODIR-5K via `kagglehub` (cached) and runs the conversion. On Kaggle
the mounted copy is used, so nothing is downloaded. Credentials (`KAGGLE_USERNAME`,
`KAGGLE_KEY`) come from `config.get_secret` (Colab Secrets → env vars → gitignored
`secrets.json`).

In [6]:
# === Dataset: ODIR-5K -> 5-class train/val/test ImageFolder ===
# Source: https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k
#
# kagglehub pulls the ODIR-5K archive once and caches it; odir_prep.prepare() then builds
# the 5-class ImageFolder the rest of the pipeline reads and sets CONFIG["data_root"].
#   - On Kaggle: the data is already mounted under /kaggle/input -> nothing to download.
#   - On Colab / locally: pulled via kagglehub. Credentials come from config.get_secret
#     (Colab Secrets -> env vars -> a gitignored secrets.json), looking for KAGGLE_USERNAME
#     and KAGGLE_KEY. Create a token at kaggle.com -> Settings -> "Create New Token".
#
# If your ODIR-5K copy uses a different Kaggle slug or mount path, set KAGGLE_SLUG / the
# mount candidates below, or call odir_prep.prepare(raw_root=<your folder>) directly.
import os, config, odir_prep
from config import CONFIG

KAGGLE_SLUG = "andrewmvd/ocular-disease-recognition-odir5k"


def download_odir():
    for mount in ("/kaggle/input/ocular-disease-recognition-odir5k",
                  "/kaggle/input/odir5k"):
        if os.path.isdir(mount):
            return mount
    os.environ.setdefault("KAGGLEHUB_CACHE", os.path.join(config.BASE_DIR, "kagglehub"))
    for var in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        if var not in os.environ:
            try:
                os.environ[var] = config.get_secret(var)
            except KeyError:
                pass
    import kagglehub
    return kagglehub.dataset_download(KAGGLE_SLUG)


_raw = download_odir()
odir_prep.prepare(raw_root=_raw)   # -> builds ImageFolder, sets CONFIG["data_root"]
print("data_root :", CONFIG["data_root"])
print("classes   :", config.fundus_class_names(refresh=True))

100%|██████████| 1.62G/1.62G [00:11<00:00, 154MB/s]

Extracting files...


[odir_prep] label table: /content/drive/MyDrive/lgmd/kagglehub/datasets/andrewmvd/ocular-disease-recognition-odir5k/versions/2/full_df.csv
[odir_prep] indexed 8000 images under /content/drive/MyDrive/lgmd/kagglehub/datasets/andrewmvd/ocular-disease-recognition-odir5k/versions/2
[odir_prep] built /content/drive/MyDrive/lgmd/cache/odir5k_imagefolder: 5324 images across 5 classes
    AMD        train= 186 val= 40 test= 40 (total 266)
    Cataract   train= 205 val= 44 test= 44 (total 293)
    Diabetes   train=1126 val=241 test=241 (total 1608)
    Glaucoma   train= 199 val= 43 test= 42 (total 284)
    Normal     train=2011 val=431 test=431 (total 2873)
data_root : /content/drive/MyDrive/lgmd/cache/odir5k_imagefolder
classes   : ['AMD', 'Cataract', 'Diabetes', 'Glaucoma', 'Normal']


## 1. Train the fundus backbone (run once)

LGMD explains a *trained* classifier, so first fine-tune DenseNet-121 to the 5-way fundus
head. This caches `densenet121_odir.pt`; rerun only when you change `n_per_class` or the
training knobs. Everything below loads these weights.

In [7]:
import train_backbone

# One-time fine-tune on n_per_class images/class (default 2000; lower for a smoke run).
if not os.path.exists(CONFIG["backbone_weights"]):
    train_backbone.train()
else:
    print("backbone weights already cached:", CONFIG["backbone_weights"])

[warn] class idx 0: only 186 train images (< 2000).
[warn] class idx 1: only 205 train images (< 2000).
[warn] class idx 2: only 1126 train images (< 2000).
[warn] class idx 3: only 199 train images (< 2000).
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 157MB/s]
epoch 1/15: 100%|██████████| 59/59 [02:01<00:00,  2.06s/it, loss=1.66]


epoch 1: train_loss=0.9163  val_acc=0.6008
  [save] new best val_acc=0.6008 -> densenet121_odir.pt


epoch 2/15: 100%|██████████| 59/59 [01:59<00:00,  2.03s/it, loss=0.404]


epoch 2: train_loss=0.7164  val_acc=0.7259
  [save] new best val_acc=0.7259 -> densenet121_odir.pt


...

epoch 13: train_loss=0.2123  val_acc=0.6133


epoch 14/15: 100%|██████████| 59/59 [01:58<00:00,  2.01s/it, loss=1.47]


epoch 14: train_loss=0.1966  val_acc=0.7372


epoch 15/15: 100%|██████████| 59/59 [02:00<00:00,  2.04s/it, loss=0.292]


epoch 15: train_loss=0.3206  val_acc=0.7509
done. best val_acc=0.7597; weights at /content/drive/MyDrive/lgmd/cache/densenet121_odir.pt


## 2. Data — the active fundus class

In [1]:
print("Fundus classes:", config.fundus_class_names())
idx = config.select_class(CONFIG["class_name"])   # active class (default: cataract)
print(f"Active class: {CONFIG['class_name']!r}  (label index {idx})")

NameError: name 'config' is not defined

In [ ]:
cls, seed = CONFIG["class_name"], CONFIG["seed"]
train_imgs = data_utils.load_class_images(cls, CONFIG["n_train"], "train_dir", seed)
val_imgs   = data_utils.load_class_images(cls, CONFIG["n_val"],   "val_dir",   seed)
print(f"{cls}: train {len(train_imgs)}  val {len(val_imgs)}")

## 3. Backbone + encoder activations (cached)

In [2]:
# Backbone and CLIP share this single 224x224 crop on purpose. Reading the same pixels
# makes their 7x7 spatial grids line up cell-for-cell, so the concept maps stay spatially
# accurate. That is why preprocessing is one shared step here, not separate per-model crops.

model, transform = model_utils.load_backbone()
Z_train = utils.cached(os.path.join(CDIR, cache_name("Z_train", ".pt", "data", "model")),
    lambda: model_utils.extract_activations(model, transform, train_imgs, desc="train activations"))
Z_val = utils.cached(os.path.join(CDIR, cache_name("Z_val", ".pt", "data", "model")),
    lambda: model_utils.extract_activations(model, transform, val_imgs, desc="val activations"))
A_train, A_val = lgmd.unfold(Z_train), lgmd.unfold(Z_val)
print(bb, "| Z_train:", tuple(Z_train.shape), " A_train:", tuple(A_train.shape))

NameError: name 'model_utils' is not defined

## 4. Concept vocabulary (stored table) + CLIP filtering

Candidate concepts are read from the static `concept_vocab.json` table (keyed by class,
~50 candidates each), then passed through the paper's two-stage filter. Reproducible and
offline — no LLM call.

In [ ]:
# get_concepts applies the paper's two-stage filter to the stored vocabulary:
#   Lexical (suppl. A1.3): keep 1-4 word concepts; drop generic filler or concepts that
#     just repeat the class name, unless they carry a clinical visual attribute
#     (lesion, color, optic-disc / vessel / macula location, texture, ...).
#   CLIP semantic (suppl. A1.4): rank survivors by BiomedCLIP similarity to the class
#     images, then greedily keep diverse ones, dropping any > 0.80 similar to one kept.

clip = clip_maps.CLIP()
# Pass train_imgs so stage-2 filtering ranks concepts by BiomedCLIP relevance to the
# class image prototype before diversity dedup (suppl. A1.4).
concept_list = concept_mod.get_concepts(clip, images=train_imgs)
print(f"{len(concept_list)} concepts:", concept_list)

## 5. Localized CLIP similarity maps S (cached — most expensive stage)

In [ ]:
S_train = utils.cached(os.path.join(CDIR, cache_name("S_train", ".pt", "data", "con", "clip")),
    lambda: clip_maps.build_S(train_imgs, concept_list, clip))
print("S_train:", tuple(S_train.shape))

## 6. Fit the semantic concept basis W (PGD)

In [ ]:
W = utils.cached(os.path.join(CDIR, cache_name("W", ".pt", "data", "model", "con", "clip", "pgd")),
    lambda: lgmd.fit_basis(A_train, S_train))
print("W:", tuple(W.shape))

## 7. Inference on val + predictive preservation / faithfulness

In [3]:
label = CONFIG["class_index"]

# Paper Sec 4: concept analysis is performed on correctly classified validation samples.
orig_logits_full = model_utils.logits_from_Z(model, Z_val)
keep = orig_logits_full.argmax(-1) == label
print(f"correctly classified val: {int(keep.sum())}/{len(keep)}")
Z_val = Z_val[keep]
val_imgs = [im for im, k in zip(val_imgs, keep.tolist()) if k]
A_val = lgmd.unfold(Z_val)
orig_logits = orig_logits_full[keep]

S_hat = lgmd.infer(A_val, W)                     # closed-form + PGD (Eqs. 4-5)
A_hat = lgmd.reconstruct(S_hat, W, Z_val.shape)

def _lgmd_metrics():
    recon_logits = model_utils.logits_from_Z(model, A_hat)
    return {
        **metrics.predictive_preservation(orig_logits, recon_logits, label),
        "kl": metrics.kl_logits(orig_logits, recon_logits),
        "recon_err": metrics.recon_error(A_val, lgmd.unfold(A_hat)),
    }

lgmd_metrics = utils.cached_json(
    os.path.join(RDIR, cache_name("lgmd_metrics", ".json", "data", "model", "con", "clip", "pgd", "infer")),
    _lgmd_metrics)
print("LGMD:", lgmd_metrics)

NameError: name 'CONFIG' is not defined

## 8. Baseline comparison (ICE, CRAFT, FACE vs LGMD) — Acc + C-Ins

In [ ]:
R = len(concept_list)   # basis columns = number of concepts
face_head = lambda a: model_utils.classify_pooled(model, a.to(DEVICE))
head_fn = lambda Z: model_utils.logits_from_Z(model, Z)

def _comparison():
    # ICE, CRAFT, and FACE differ only in how the basis W is learned (NMF, recursive
    # NMF, and KL-regularized NMF respectively). Everything else is held identical:
    # backbone, preprocessing, splits, r, and the non-negative inference step (Sec 4).
    bases = {
        "ICE":   baselines.fit_ice(A_train, R),
        "CRAFT": baselines.fit_craft(A_train, R),
        "FACE":  baselines.fit_face(A_train, R, face_head, Z_train.shape),
        "LGMD":  W,
    }
    out = {}
    for name, Wb in bases.items():
        Sb = lgmd.infer(A_val, Wb)
        Ab = lgmd.reconstruct(Sb, Wb, Z_val.shape)
        lg = model_utils.logits_from_Z(model, Ab)
        cur = metrics.faithfulness_curves(Sb, Wb, Z_val.shape, head_fn, label)
        pp = metrics.predictive_preservation(orig_logits, lg, label)
        out[name] = {
            "Acc": pp["recon_acc"],
            # C-Ins is the normalized area under the concept-insertion curve (Sec 4.2):
            # how quickly the correct-class prediction is restored as top-ranked
            # concepts are added back in.
            "C-Ins": metrics.insertion_auc(cur["insertion"]),
            "agreement": pp["agreement"],
            "kl": metrics.kl_logits(orig_logits, lg),
            "recon_err": metrics.recon_error(A_val, lgmd.unfold(Ab)),
        }
    return out

rows = utils.cached_json(
    os.path.join(RDIR, cache_name("comparison", ".json",
                                  "data", "model", "con", "clip", "pgd", "infer", "base", "cins")),
    _comparison)

# Print the results table
import pandas as pd
df_comp = pd.DataFrame(rows).T
display(df_comp)
print(json.dumps(rows, indent=2))

## 9. Faithfulness curves (C-Deletion / C-Insertion) for LGMD

In [4]:
import matplotlib.pyplot as plt
curves = metrics.faithfulness_curves(S_hat, W, Z_val.shape, head_fn, label)
plt.plot(curves["deletion"], label="C-Deletion")
plt.plot(curves["insertion"], label="C-Insertion")
plt.xlabel("# concepts removed / added"); plt.ylabel("true-class prob"); plt.legend()
plt.savefig(os.path.join(RDIR, f"faithfulness_{bb}.png"), dpi=120, bbox_inches="tight")
plt.show()

NameError: name 'metrics' is not defined

## 10. Concept heatmap overlays (Eq. 6)

In [ ]:
viz.save_concept_overlays(val_imgs, S_hat, concept_list, out_name=f"overlays_{bb}.png", max_images=4)

## 11. Ablation — closed-form vs closed-form+PGD coefficient estimation (Table 4)

In [ ]:
# Closed-form (Eq. 4) vs closed-form + PGD (Eq. 5). Both preserve accuracy; the
# hybrid slightly improves C-Ins and lowers reconstruction MSE (paper Sec 4.5). Cached on Drive.
def _ablation():
    out = {}
    for mode, refine in [("Closed-form", False), ("Closed-form+PGD", True)]:
        Sh = lgmd.infer(A_val, W, refine=refine)
        Ah = lgmd.reconstruct(Sh, W, Z_val.shape)
        lg = model_utils.logits_from_Z(model, Ah)
        pp = metrics.predictive_preservation(orig_logits, lg, label)
        cur = metrics.faithfulness_curves(Sh, W, Z_val.shape, head_fn, label)
        out[mode] = {
            "Acc": pp["recon_acc"],
            "C-Ins": metrics.insertion_auc(cur["insertion"]),
            "MSE": metrics.mse(A_val, lgmd.unfold(Ah)),
        }
    return out

abl = utils.cached_json(
    os.path.join(RDIR, cache_name("ablation_table4", ".json",
                                  "data", "model", "con", "clip", "pgd", "infer", "cins")),
    _ablation)
print(json.dumps(abl, indent=2))

## 12. Concept activation scores: before (CLIP init) vs after optimization (Fig 4)

In [ ]:
import importlib
import viz

# Reload the viz module to apply changes from the disk
importlib.reload(viz)
print("Successfully reloaded viz.py")

In [ ]:
# "Before" = raw CLIP-similarity init on val images (expensive: cached); "after" = S_hat.
# Depends on backbone too: val images are filtered to correctly classified samples.
S_val_clip = utils.cached(os.path.join(CDIR, cache_name("S_val", ".pt", "data", "model", "con", "clip")),
    lambda: clip_maps.build_S(val_imgs, concept_list, clip))   # same row order as A_val / S_hat
viz.plot_score_distributions(S_val_clip, S_hat, out_name=f"fig4_score_distributions_{bb}.png")


In [ ]:
# Single clean row: input image + its top named-concept heatmaps.
viz.plot_concept_heatmaps(val_imgs, S_hat, concept_list, img_index=0, top_k=3,
                          out_name=f"fig1_heatmaps_{bb}.png")

In [ ]:
# Fig 3 (example-patch style): each concept shown by its top-activating image regions.
# ICE localizes on the pre-pool feature map -> red region outlines ('red circle regions');
# CRAFT/FACE pool spatially -> cropped exemplar patches; LGMD = cropped patches + names.
# Re-fits the baseline bases (expensive) so the cell is self-contained.
import lgmd
bases = {
    "ICE":   baselines.fit_ice(A_train, R),
    "CRAFT": baselines.fit_craft(A_train, R),
    "FACE":  baselines.fit_face(A_train, R, face_head, Z_train.shape),
    "LGMD":  W,
}
method_maps = {name: lgmd.infer(A_val, Wb) for name, Wb in bases.items()}
styles = {"ICE": "contour", "CRAFT": "crop", "FACE": "crop", "LGMD": "crop"}
for name, S in method_maps.items():
    viz.plot_concept_patches(
        val_imgs, S,
        concepts=concept_list if name == "LGMD" else None,
        style=styles[name], top_concepts=3, n_patches=3, title=name,
        out_name=f"fig3_{name.lower()}_{bb}.png")

In [ ]:
bases = {
    "ICE":   baselines.fit_ice(A_train, R),
    "CRAFT": baselines.fit_craft(A_train, R),
    "FACE":  baselines.fit_face(A_train, R, face_head, Z_train.shape),
    "LGMD":  W,
}
method_maps = {name: lgmd.infer(A_val, Wb) for name, Wb in bases.items()}
viz.plot_baseline_comparison(val_imgs, method_maps, n_images=3,
                             concepts_by_method={"LGMD": concept_list},
                             out_name=f"fig3_baselines_{bb}.png")

## 13. Run all 7 fundus classes

Everything above runs the active single class. This driver runs all classes: it loads the
backbone + BiomedCLIP once, then loops over `config.fundus_class_names()`, caching every
artifact per class. A class that errors is printed and skipped. Concept-overlay figures are
produced only for `config.FIGURE_CLASSES`. Pass a subset to `run_all([...])` for fewer.

In [ ]:
# No runtime restart needed after a git pull: reload the changed modules IN PLACE,
# in dependency order. Reload `concepts` BEFORE `runner` — runner imports concepts, so
# reloading runner second re-binds it to the new concepts code. (config is NOT reloaded:
# that would swap out the CONFIG object this notebook already holds; the new
# concept_skip_if_short flag defaults to True via CONFIG.get, so no reload is required.)
import importlib, concepts as concept_mod, runner, viz
importlib.reload(concept_mod)
importlib.reload(runner)
importlib.reload(viz)

# Run every class (or e.g. runner.run_all(["Glaucoma", "AMD"]) for a subset).
# A class whose concepts come up short (or that errors otherwise) is SKIPPED, not fatal:
# `failures` maps each skipped class -> reason and is printed as a numbered list below.
results, agg, failures = runner.run_all()

import pandas as pd
print("Per-method mean over", len(results), "classes (Acc, C-Ins):")
display(pd.DataFrame(agg).T)

# Per-class breakdown: one row per retinal-disease class (+ an Average row), one column
# per method, for each metric. Shown as a DataFrame and saved as a table image (best value
# bold / second-best underlined) via viz.render_metric_table.
methods = ("LGMD", "FACE", "ICE", "CRAFT")
for metric in ("Acc", "C-Ins"):
    tbl = runner.metric_table(results, metric, methods=methods)
    print(f"\nPer-class {metric} (rows = classes):")
    display(pd.DataFrame(tbl[bb]).T.reindex(columns=list(methods)))
    viz.render_metric_table(
        tbl, out_name=f"table_perclass_{metric.lower().replace('-', '')}_{bb}.png",
        methods=methods, backbones=(bb,), higher_is_better=True,
        title=f"Per-class {metric} — {bb}")

In [ ]:
# Per-class performance across ALL metrics used, one table per retinal-disease class
# (methods as rows; Acc / C-Ins / agreement are higher-better, kl / recon_err lower-better).
# Best method per metric is bold, second-best underlined; each table is saved to viz_dir as
# table_<class>_metrics.png.
import pandas as pd
metrics_all = ["Acc", "C-Ins", "agreement", "kl", "recon_err"]
for cls, r in results.items():
    comp = r.get("comparison")
    if not comp:
        continue
    print(f"\n=== {cls}: methods x metrics ===")
    display(pd.DataFrame(comp).T.reindex(index=list(methods), columns=metrics_all))
    viz.render_class_metric_table(cls, comp, metrics=metrics_all, methods=methods,
                                  out_name=f"table_{cls.lower()}_metrics.png")

### Recover skipped classes
If `run_all` skipped a class (e.g. *"Only 2/25 concepts survived filtering for 'Cataract'"*),
top up that class's candidate vocabulary and re-evaluate **only** that class — every class that
already succeeded stays cached.

Set `failed_idx` to the class's number from the list printed above, and `new_concepts` to extra
2-4 word clinical phrases (distinct enough to survive the CLIP dedup). Re-run as often as needed.

In [ ]:
# === Recover a skipped class: add concepts to its vocab, then re-evaluate ===
import importlib, runner, concepts as concept_mod
importlib.reload(concept_mod); importlib.reload(runner)

# 1) which skipped class to fix (its number from the list printed by run_all above)
failed_idx = 0

# 2) extra candidate concepts for that class (2-4 word clinical phrases)
new_concepts = [
    "cortical spoke opacities",
    "posterior subcapsular cataract",
    "nuclear sclerosis",
    "lens protein clumping",
    # ... add as many as needed so >= r (=25) survive filtering
]

failed = list(failures)                          # skipped class names, in printed order
if not failed:
    print("Nothing was skipped - no class to recover.")
else:
    cls = failed[failed_idx]
    print(f"Recovering [{failed_idx}] {cls!r} ...")
    concept_mod.add_concepts(cls, new_concepts)  # update concept_vocab.json + refresh cache key

    res, _, refail = runner.run_all([cls])       # re-evaluate just this class
    results.update(res)                          # merge into the full results
    failures.pop(cls, None)
    failures.update(refail)                      # still short? it stays for another pass

    import pandas as pd
    print("Remaining skipped classes:", list(failures))
    print("Per-method mean over", len(results), "classes (Acc, C-Ins):")
    display(pd.DataFrame(runner.aggregate(results)).T)

import importlib, runner, config
importlib.reload(runner)

# Render the qualitative figures for ALL 5 retinal-disease classes. Order matters: the
# figure functions place classes top-to-bottom in the order given, so we put AMD and
# Diabetes (the diabetic_retinopathy class; its folder is "Diabetes") first. Remaining
# classes follow in their default order. Artifacts are cached per class, so reruns are cheap.
_all = config.fundus_class_names()
_priority = ["AMD", "Diabetes"]
_ordered = [c for c in _priority if c in _all] + [c for c in _all if c not in _priority]
fig_data = runner.make_figures(_ordered)

In [ ]:
import importlib, runner, config
importlib.reload(runner)

# Render the qualitative figures for ALL 5 retinal-disease classes (not just cataract):
# each figure gets one row/block per class. Artifacts are cached per class, so reruns are
# cheap. Pass an explicit list instead to restrict, e.g. make_figures(["Cataract", "AMD"]).
fig_data = runner.make_figures(config.fundus_class_names())